## Cell 1 — Install dependencies & mount Google Drive

In [ ]:
import subprocess, sys, importlib, os

def _pip(pkg):
    name = pkg.split('[')[0].replace('-', '_')
    if importlib.util.find_spec(name) is None:
        print(f'Installing {pkg}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
    else:
        print(f'\u2713 {pkg} already installed')

for pkg in ['google-api-python-client', 'google-auth-httplib2',
            'google-auth-oauthlib', 'openpyxl', 'pandas']:
    _pip(pkg)

try:
    import google.colab
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    else:
        print('\u2713 Google Drive already mounted')
except ModuleNotFoundError:
    print('Not in Colab — skipping Drive mount')

print('\nReady.')

Installing google-api-python-client...
✓ google-auth-httplib2 already installed
✓ google-auth-oauthlib already installed
✓ openpyxl already installed
✓ pandas already installed
Mounted at /content/drive

Ready.


## Cell 2 — Imports and path configuration

> **Edit `BASE` if your folder structure is different.**

In [ ]:
import json, time
import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

# ============================================================
# PATHS - Edit these as needed
# ============================================================
BASE = "BASE"
GFORM_DIR = f'{BASE}/Gform_Preparation'
CREDS_FILE = f'{BASE}/credentials.json'
TOKEN_FILE = f'{BASE}/token.json'
LINKS_OUT = f'{GFORM_DIR}/form_links.json'

# ============================================================
# EVAL FILES - The 2 eval files produced by Phase 2, in order
# ============================================================
EVAL_FILES = [
    ('Group B - Part 1', f'{GFORM_DIR}/Eval_Part1.xlsx'),
    ('Group B - Part 2', f'{GFORM_DIR}/Eval_Part2.xlsx'),
]

SCOPES = [
    'https://www.googleapis.com/auth/forms.body',
    'https://www.googleapis.com/auth/drive',
]

print('Paths configured.')
print('Eval files:')
for label, path in EVAL_FILES:
    exists = '\u2713' if os.path.exists(path) else '\u2717 NOT FOUND'
    print(f'  {exists} {label}: {os.path.basename(path)}')

## Cell 3 — Form content: guidelines and score options

In [ ]:
GUIDELINES = """EVALUATION GUIDELINES: ODIA SPELLING AND GRAMMATICAL ERROR TASK

You are evaluating responses for an Odia spelling and grammatical error detection task. Each instance includes:

Evaluate only the given target response. Do not introduce new errors.

COMPONENT 1 - ERROR EXISTS [0 / 1]
1 : Correctly identifies whether an error is present.
0 : Incorrect.

COMPONENT 2 - ERROR CATEGORY [0 / 1]

Categories:
- Script Normalization
- Spelling & Typographical Errors
- Grammatical Errors
- Code-Mixing / Wrong Language
- Correct Sentence / No Errors

1 : Correct category (specific or parent-level acceptable).
0 : Incorrect.

If no error exists, category must be "Correct Sentence / No Errors".

COMPONENT 3 - SPAN + DESCRIPTION [0 / 1 / 2]
2 : Exact span AND fully correct explanation (what, why, correct form).
1 : Partial span and/or incomplete explanation.
0 : Wrong span or hallucinated/irrelevant explanation.

If no error exists, span must be empty and description should state no error.

COMPONENT 4 - CORRECTED SENTENCE [0 / 1 / 2]
2 : Fully correct, fluent, no new errors.
1 : Mostly correct with minor issues.
0 : Incorrect, introduces errors, or changes meaning.

If no error exists, output must match the source sentence."""

TASK_NOTICE = """This form contains 50 tasks - one per page.

All questions on every page are mandatory. You cannot proceed to the next page without answering all questions on the current page.

You can navigate back to previous pages to review or change your responses at any time before final submission."""

C1_OPTIONS = [
    '1 - Correctly identified',
    '0 - Incorrect identification',
]

C2_OPTIONS = [
    '1 - Correct category',
    '0 - Incorrect category',
]

C3_OPTIONS = [
    '2 - Exact span AND fully correct explanation',
    '1 - Partial span and/or incomplete explanation',
    '0 - Wrong span or hallucinated/irrelevant explanation',
]

C4_OPTIONS = [
    '2 - Fully correct, fluent, no new errors',
    '1 - Mostly correct with minor issues',
    '0 - Incorrect, introduces errors, or changes meaning',
]

ODIA_PROF_OPTIONS = [
    'Native / Fluent',
    'Advanced',
    'Intermediate',
    'Basic',
    'None',
]

Content configured.


## Cell 4 — Google OAuth authentication

> **First run:** A URL will print. Open it, sign in, copy the code, paste it back here.
> **Subsequent runs:** Skipped automatically (token cached).

In [ ]:
def get_creds():
    creds = None
    if os.path.exists(TOKEN_FILE):
        creds = Credentials.from_authorized_user_file(TOKEN_FILE, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(CREDS_FILE, SCOPES)
            try:
                import google.colab
                flow.redirect_uri = 'urn:ietf:wg:oauth:2.0:oob'
                auth_url, _ = flow.authorization_url(prompt='consent')
                print('\nOpen this URL in your browser and sign in:')
                print(auth_url)
                code = input('\nPaste the authorization code here: ').strip()
                flow.fetch_token(code=code)
                creds = flow.credentials
            except ModuleNotFoundError:
                creds = flow.run_local_server(port=0)
        with open(TOKEN_FILE, 'w') as f:
            f.write(creds.to_json())
    return creds

creds     = get_creds()
forms_svc = build('forms', 'v1', credentials=creds)
print('\u2713 Authenticated and Forms API ready.')

✓ Authenticated and Forms API ready.


## Cell 5 — Data loader: read one eval xlsx file

In [ ]:
def load_eval_file(path):
    """
    Reads an eval xlsx produced by Phase 2.
    Phase 2 writes 3 header rows: row1=zone headers, row2=evaluator headers, row3=col names.
    Data starts at row 4.  Use header=2 (0-indexed) to get row 3 as columns.
    Returns a list of task dicts with only the fields needed for the form.
    """
    df = pd.read_excel(path, header=2, dtype=str).fillna('')

    # Keep only the columns we need -- drop score columns
    needed = ['#', 'Source Sentence', 'Has Errors', 'Error Span',
              'Annotated Category', 'Description', 'Corrected Sentence']
    df = df[[c for c in needed if c in df.columns]]

    tasks = []
    for _, row in df.iterrows():
        serial = str(row.get('#', '')).strip()
        if not serial or serial == 'nan':
            continue
        tasks.append({
            'serial':      serial,
            'source':      row.get('Source Sentence', '').strip(),
            'has_errors':  row.get('Has Errors', '').strip(),
            'span':        row.get('Error Span', '').strip(),
            'category':    row.get('Annotated Category', '').strip(),
            'description': row.get('Description', '').strip(),
            'corrected':   row.get('Corrected Sentence', '').strip(),
        })
    return tasks


# Quick verification
label0, path0 = EVAL_FILES[0]
sample = load_eval_file(path0)
print(f'Sample load from "{label0}":')
print(f'  Tasks found: {len(sample)}')
print(f'  Task 1 serial : {sample[0]["serial"]}')
print(f'  Task 1 source : {sample[0]["source"][:60]}...')
print(f'  Task 1 has_errors : {sample[0]["has_errors"]}')

Sample load from "Group A — Part 1":
  Tasks found: 50
  Task 1 serial : 1
  Task 1 source : ମୋହନ ଭାଗଓ୍ବତ୍ କହିଛନ୍ତି, "ଏବେ ହେବ ରାମଙ୍କ କାମ‌।"...
  Task 1 has_errors : True


## Cell 6 — Form item builders

In [ ]:
def radio_question(title, description, options, required=True):
    """Returns a createItem request dict for a radio question."""
    return {
        'questionItem': {
            'question': {
                'required': required,
                'choiceQuestion': {
                    'type': 'RADIO',
                    'options': [{'value': v} for v in options],
                },
            }
        },
        'title':       title,
        'description': description,
    }


def text_question(title, description='', required=True):
    """Returns a createItem request dict for a short-text question."""
    return {
        'questionItem': {
            'question': {
                'required': required,
                'textQuestion': {'paragraph': False},
            }
        },
        'title':       title,
        'description': description,
    }


def text_display(title, description):
    """Returns a non-question text block (for source sentence display)."""
    return {
        'textItem': {},
        'title':       title,
        'description': description,
    }


def page_break(title, description=''):
    """Returns a page break item."""
    return {
        'pageBreakItem': {},
        'title':       title,
        'description': description,
    }


def build_task_items(task, total_tasks=50):
    """
    Returns a list of form items for one task page.
    Clean card layout:
      - Page break titled 'Task N of 50'
      - Source sentence display block
      - Separator then C1, C2, C3, C4 questions
    Model output embedded directly in each question title/description.
    Nothing identifying (Sentence_ID, LLM, Gold Category) exposed.
    """
    n = task['serial']

    # Format span and description for display
    span_display = task['span'] if task['span'] else '(none)'
    desc_display = task['description'] if task['description'] else '(none)'

    items = []

    # Page break — clean title
    items.append(page_break(
        title=f'Task {n} of {total_tasks}',
    ))

    # Source sentence — displayed as a clean text block
    items.append(text_display(
        title='Source Sentence',
        description=task['source'],
    ))

    # Detected info separator
    items.append(text_display(
        title='Detected Information',
        description=(
            f'Error detected : {task["has_errors"]}\n'
            f'Error span     : {span_display}\n'
            f'Category       : {task["category"]}\n'
            f'Description    : {desc_display}\n'
            f'Corrected      : {task["corrected"]}'
        ),
    ))

    # C1 — Error existence
    items.append(radio_question(
        title='C1 — Is the error detection correct?',
        description=f'Error detected: {task["has_errors"]}',
        options=C1_OPTIONS,
    ))

    # C2 — Span + description
    items.append(radio_question(
        title='C2 — Is the error span and description correct?',
        description=(
            f'Span: {span_display}\n'
            f'Description: {desc_display}'
        ),
        options=C3_OPTIONS,   # 2/1/0 scale
    ))

    # C3 — Category
    items.append(radio_question(
        title='C3 — Is the error category correct?',
        description=f'Annotated category: {task["category"]}',
        options=C2_OPTIONS,   # 1/0 scale
    ))

    # C4 — Corrected sentence
    items.append(radio_question(
        title='C4 — Is the corrected sentence correct?',
        description=f'Corrected: {task["corrected"]}',
        options=C4_OPTIONS,   # 2/1/0 scale
    ))

    return items


print('Item builders ready.')

Item builders ready.


## Cell 7 — Page 1 builder (profile + guidelines + notice)

In [ ]:
def build_page1_items():
    """Returns all items for page 1: profile, guidelines, task notice."""
    items = []

    # ── Section 1: Annotator profile ──────────────────────────────────────
    items.append(text_display(
        title='Section 1 — Annotator Information',
        description='Please fill in your details below. All fields are mandatory.',
    ))
    items.append(text_question('Full Name'))
    items.append(text_question('Age'))
    items.append(text_question('Email ID'))
    items.append(text_question(
        title='Contact Information',
        description='Phone number or an alternate email address.',
    ))
    items.append(radio_question(
        title='Odia Proficiency',
        description='Select your level of proficiency in reading and writing Odia.',
        options=ODIA_PROF_OPTIONS,
    ))
    items.append(radio_question(
        title='I voluntarily agree to participate in this evaluation study.',
        description='Your responses will be used solely for academic research purposes.',
        options=['Yes, I agree', 'No, I do not agree'],
    ))

    # ── Section 2: Evaluation guidelines ──────────────────────────────────
    items.append(text_display(
        title='Section 2 — Evaluation Guidelines',
        description=GUIDELINES,
    ))

    # ── Section 3: Task notice ─────────────────────────────────────────────
    items.append(text_display(
        title='Section 3 — Task Instructions',
        description=TASK_NOTICE,
    ))

    return items


print('Page 1 builder ready.')

Page 1 builder ready.


## Cell 8 — Form creator: assembles and uploads one full form

In [ ]:
def build_requests(items_list):
    """Converts a list of item dicts into batchUpdate createItem requests."""
    return [
        {'createItem': {'item': item, 'location': {'index': idx}}}
        for idx, item in enumerate(items_list)
    ]


def create_form(label, tasks):
    """
    Creates one complete Google Form.
    label  : human-readable label e.g. 'Group A — Part 1'
    tasks  : list of task dicts from load_eval_file()
    Returns form_id and form_url.
    """
    total_tasks = len(tasks)

    # 1. Create empty form shell
    form = forms_svc.forms().create(body={
        'info': {
            'title':         'Error Detection Evaluation Task',
            'documentTitle': f'EvalForm_{label.replace(" ", "_").replace("—","-")}',
        }
    }).execute()
    form_id = form['formId']
    print(f'  Form created: {form_id}')

    # 2. Assemble all items
    all_items = []

    # Page 1 — profile + guidelines + notice
    all_items.extend(build_page1_items())

    # Pages 2..N+1 — one task per page
    for task in tasks:
        all_items.extend(build_task_items(task, total_tasks=total_tasks))

    # 3. Upload in batches (10 items per batch — conservative for Odia text)
    requests = build_requests(all_items)
    BATCH    = 10
    total_batches = (len(requests) + BATCH - 1) // BATCH

    for batch_num, i in enumerate(range(0, len(requests), BATCH), start=1):
        batch = requests[i: i + BATCH]
        forms_svc.forms().batchUpdate(
            formId=form_id,
            body={'requests': batch},
        ).execute()
        print(f'  Batch {batch_num}/{total_batches} uploaded ({len(batch)} items)')
        time.sleep(0.8)   # rate limit buffer

    # 4. Minimal settings (no quiz mode, no irrelevant fields)
    forms_svc.forms().batchUpdate(
        formId=form_id,
        body={'requests': [{
            'updateSettings': {
                'settings': {'quizSettings': {'isQuiz': False}},
                'updateMask': 'quizSettings',
            }
        }]},
    ).execute()

    form_url = f'https://docs.google.com/forms/d/{form_id}/viewform'
    print(f'  Done: {form_url}')
    return form_id, form_url


print('Form creator ready.')

Form creator ready.


## Cell 9 — Generate all 4 forms

In [ ]:
form_links = []

for form_num, (label, path) in enumerate(EVAL_FILES, start=1):
    print(f'\n{"-"*60}')
    print(f'Form {form_num}/4 — {label}')
    print(f'Source file: {os.path.basename(path)}')
    print(f'{"-"*60}')

    tasks = load_eval_file(path)
    print(f'Tasks loaded: {len(tasks)}')

    form_id, form_url = create_form(label, tasks)

    form_links.append({
        'form_num':   form_num,
        'label':      label,
        'source_file': os.path.basename(path),
        'form_id':    form_id,
        'form_url':   form_url,
    })

    time.sleep(2.0)   # pause between forms

# Save links
with open(LINKS_OUT, 'w', encoding='utf-8') as f:
    json.dump(form_links, f, ensure_ascii=False, indent=2)
print(f'\n\u2713 Saved form links to {LINKS_OUT}')

## Cell 10 — Print all form links

In [ ]:
print('\n' + '='*65)
print('ALL 4 FORM LINKS — share with annotators')
print('='*65)
for info in form_links:
    print(f"\nForm {info['form_num']}: {info['label']}")
    print(f"  Source : {info['source_file']}")
    print(f"  URL    : {info['form_url']}")
print('\n' + '='*65)
print('REMINDER:')
print('  - Each form has 50 tasks + 1 profile page = 51 pages total')
print('  - All questions are mandatory')
print('  - Annotators can navigate back to change answers')
print('  - To collect responses: Forms UI > Responses tab > Download CSV')
print('  - Links also saved to:', LINKS_OUT)
print('='*65)